# Neighbor2Neighbor — Implementation / 구현

**Paper**: Huang, T., Li, S., Jia, X., Lu, H., Liu, J. (2021). *Neighbor2Neighbor: Self-Supervised Denoising from Single Noisy Images*. CVPR 2021.

## Overview / 개요

**한국어** — 본 노트북은 Nb2Nb의 핵심 building block을 시연한다:
1. **Random neighbour sub-sampler** \\( G = (g_1, g_2) \\) 구현 — 각 \\( 2\times 2 \\) 셀에서 인접 픽셀 두 개 무작위 선택
2. **잡음의 statistical near-independence** + 신호의 **near-equality** 검증
3. **작은 CNN + N2N 손실 + regulariser** (Eq. 7) 학습
4. Compare against frame-by-frame BM3D and noisy baseline
5. Regulariser ablation (\\( \gamma = 0 \\) vs \\( \gamma = 2 \\))

Full Nb2Nb은 ImageNet 50k 패치에 UNet/RRG로 학습; 본 노트북은 단일 영상에 작은 CNN을 짧게 학습.

**English** — Demonstrates Nb2Nb's core building blocks:
1. Random neighbour sub-sampler \\( G = (g_1, g_2) \\) — pick 2 adjacent pixels per 2×2 cell.
2. Verify noise near-independence + signal near-equality between sub-images.
3. Train tiny CNN with N2N loss + regulariser (Eq. 7).
4. Compare with BM3D and noisy baseline.
5. Regulariser ablation (γ=0 vs γ=2).

Full Nb2Nb trains UNet/RRG on 50k ImageNet patches; this notebook trains a tiny CNN on a single image.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.typing import NDArray
from skimage import data, util

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 10
torch.manual_seed(42); np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


def psnr(a: NDArray, b: NDArray, peak: float = 1.0) -> float:
    """Peak Signal-to-Noise Ratio (dB)."""
    return float(10 * np.log10(peak ** 2 / max(np.mean((a - b) ** 2), 1e-12)))

---

## Part 1: Test image and noise / 테스트 이미지

Cameraman cropped to 128×128, σ=0.10 Gaussian. / 128×128 cameraman crop, σ=0.10 Gaussian noise.

In [ ]:
clean_full = util.img_as_float(data.camera()).astype(np.float32)
clean = clean_full[64:192, 64:192]  # 128x128
sigma = 0.10
rng = np.random.default_rng(42)
noisy = clean + sigma * rng.standard_normal(clean.shape).astype(np.float32)

print(f'Image shape : {clean.shape}')
print(f'Noisy PSNR  : {psnr(noisy, clean):.2f} dB')

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(clean, cmap='gray', vmin=0, vmax=1); axes[0].set_title('clean')
axes[1].imshow(noisy, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'noisy ({psnr(noisy, clean):.2f} dB)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---

## Part 2: Random neighbour sub-sampler / 무작위 이웃 sub-sampler

Partition image into \\( 2 \times 2 \\) cells. In each cell pick two *neighbouring* positions (4 options: horizontal, vertical, or two diagonals). One goes to \\( g_1 \\), the other to \\( g_2 \\). Result: two sub-images of size \\( H/2 \times W/2 \\).

각 \\( 2 \times 2 \\) 셀에서 인접한 두 픽셀 무작위 선택 → \\( g_1, g_2 \\). 결과 sub-image 크기 \\( H/2 \times W/2 \\).

In [ ]:
# 4 neighbour pair patterns within a 2x2 cell.
# Cell positions: (0,0)=A, (0,1)=B, (1,0)=C, (1,1)=D.
# Adjacent pairs (sharing an edge): (A,B), (A,C), (B,D), (C,D).
# Following the paper (Fig. 2), we use these four patterns.
PAIR_PATTERNS = [
    ((0, 0), (0, 1)),  # A-B horizontal top
    ((0, 0), (1, 0)),  # A-C vertical left
    ((0, 1), (1, 1)),  # B-D vertical right
    ((1, 0), (1, 1)),  # C-D horizontal bottom
]


def neighbour_subsample(img: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Random neighbour sub-sampler g_1, g_2 (Nb2Nb §4.1).

    Args:
        img: Tensor of shape (B, C, H, W) where H, W are even.

    Returns:
        Two tensors of shape (B, C, H//2, W//2).
    """
    B, C, H, W = img.shape
    h, w = H // 2, W // 2
    # Random pattern index for each cell, shape (h, w).
    idx = torch.randint(0, len(PAIR_PATTERNS), (h, w), device=img.device)
    # Pre-compute reshaped view of cells: (B, C, h, 2, w, 2).
    cells = img.view(B, C, h, 2, w, 2)
    # Build offsets for each cell using PAIR_PATTERNS.
    pat_tensor = torch.tensor(PAIR_PATTERNS, device=img.device)  # (4, 2, 2): 4 patterns × 2 picks × (di, dj)
    di1, dj1 = pat_tensor[idx, 0, 0], pat_tensor[idx, 0, 1]  # (h, w)
    di2, dj2 = pat_tensor[idx, 1, 0], pat_tensor[idx, 1, 1]
    # Gather using fancy indexing.
    arange_h = torch.arange(h, device=img.device).view(h, 1).expand(h, w)
    arange_w = torch.arange(w, device=img.device).view(1, w).expand(h, w)
    g1 = cells[:, :, arange_h, di1, arange_w, dj1]
    g2 = cells[:, :, arange_h, di2, arange_w, dj2]
    return g1, g2


noisy_t = torch.from_numpy(noisy).unsqueeze(0).unsqueeze(0).to(device)
clean_t = torch.from_numpy(clean).unsqueeze(0).unsqueeze(0).to(device)
g1_n, g2_n = neighbour_subsample(noisy_t)
g1_c, g2_c = neighbour_subsample(clean_t)  # for verification only — different random pattern each call

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes[0, 0].imshow(clean, cmap='gray', vmin=0, vmax=1); axes[0, 0].set_title('clean')
axes[0, 1].imshow(g1_c[0, 0].cpu(), cmap='gray', vmin=0, vmax=1); axes[0, 1].set_title('g1(clean)')
axes[0, 2].imshow(g2_c[0, 0].cpu(), cmap='gray', vmin=0, vmax=1); axes[0, 2].set_title('g2(clean)')
axes[1, 0].imshow(noisy, cmap='gray', vmin=0, vmax=1); axes[1, 0].set_title('noisy')
axes[1, 1].imshow(g1_n[0, 0].cpu(), cmap='gray', vmin=0, vmax=1); axes[1, 1].set_title('g1(noisy)')
axes[1, 2].imshow(g2_n[0, 0].cpu(), cmap='gray', vmin=0, vmax=1); axes[1, 2].set_title('g2(noisy)')
for ax in axes.flat: ax.axis('off')
plt.tight_layout(); plt.show()

print(f'Sub-image shape: {tuple(g1_n.shape)}')

---

## Part 3: Verify near-independence of sub-image noise / 잡음의 near-independence 검증

For Gaussian iid noise, sub-image noise should be statistically independent: per-pixel correlation \\( \approx 0 \\). Signal gap \\( \boldsymbol\varepsilon = g_2(\mathbf x) - g_1(\mathbf x) \\) should have small magnitude (textured vs flat regions).

Sub-image 잡음의 pixel-wise correlation ≈ 0, 신호 차이 \\( \varepsilon \\)는 작은 magnitude.

In [ ]:
# Use one fixed random sub-sampling pattern for analysis
torch.manual_seed(0)
g1_n, g2_n = neighbour_subsample(noisy_t)
torch.manual_seed(0)
g1_c, g2_c = neighbour_subsample(clean_t)  # same pattern

noise_g1 = (g1_n - g1_c).cpu().numpy().flatten()
noise_g2 = (g2_n - g2_c).cpu().numpy().flatten()
signal_eps = (g2_c - g1_c).cpu().numpy().flatten()

print('Pixel-wise correlation between noise in g1 and g2:')
print(f'  corr(n_g1, n_g2)        : {np.corrcoef(noise_g1, noise_g2)[0, 1]:+.4f}  (≈ 0 → independent)')
print(f'  std(n_g1)                : {noise_g1.std():.4f}  (≈ σ = {sigma})')
print(f'  std(n_g2)                : {noise_g2.std():.4f}')
print(f'\nSignal gap ε = g2(clean) - g1(clean):')
print(f'  mean(|ε|)                : {np.abs(signal_eps).mean():.4f}  (small but non-zero)')
print(f'  std(ε)                   : {signal_eps.std():.4f}')
print(f'  → Theorem 1 cross term needs the regulariser to suppress.')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].scatter(noise_g1[:5000], noise_g2[:5000], s=1, alpha=0.3)
axes[0].set_xlabel('noise in g1'); axes[0].set_ylabel('noise in g2')
axes[0].set_title(f'corr ≈ {np.corrcoef(noise_g1, noise_g2)[0, 1]:+.3f}'); axes[0].grid(True)
axes[1].hist(noise_g1, bins=50, alpha=0.6, label='n_g1')
axes[1].hist(noise_g2, bins=50, alpha=0.6, label='n_g2')
axes[1].set_title('Sub-image noise distributions'); axes[1].legend(); axes[1].grid(True)
axes[2].hist(signal_eps, bins=50, color='C2', alpha=0.7)
axes[2].set_title(f'Signal gap ε  (std={signal_eps.std():.3f})'); axes[2].grid(True)
plt.tight_layout(); plt.show()

---

## Part 4: Tiny CNN architecture / 작은 CNN

3 convolutional layers, 32 channels. Standard architecture (no blind-spot, no dropout) — the key advantage of Nb2Nb is *architecture freedom*.

표준 3-conv-layer CNN. Nb2Nb의 핵심 장점은 *architecture freedom*.

In [ ]:
class TinyCNN(nn.Module):
    """Three-layer convolutional denoiser (no blind-spot, no dropout)."""

    def __init__(self, channels: int = 32) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(1, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv3 = nn.Conv2d(channels, channels, 3, padding=1)
        self.out = nn.Conv2d(channels, 1, 3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.leaky_relu(self.conv1(x), 0.1)
        h = F.leaky_relu(self.conv2(h), 0.1)
        h = F.leaky_relu(self.conv3(h), 0.1)
        # Residual connection helps in tiny single-image regime
        return self.out(h) + x

---

## Part 5: Train with N2N loss + regulariser / N2N 손실 + regulariser 학습

Loss (Eq. 7):
\\[ \mathcal L = \| f_\theta(g_1(\mathbf y)) - g_2(\mathbf y) \|^2 + \gamma \| f_\theta(g_1(\mathbf y)) - g_2(\mathbf y) - (g_1(f_\theta(\mathbf y)) - g_2(f_\theta(\mathbf y))) \|^2 \\]

Note: gradient is *stopped* on \\( g_1(f_\theta(\mathbf y)), g_2(f_\theta(\mathbf y)) \\) — only \\( f_\theta(g_1(\mathbf y)) \\) has gradient. Train two models: \\( \gamma = 0 \\) (rec only) vs \\( \gamma = 2 \\) (with reg).

두 모델 학습: γ=0 (rec only), γ=2 (reg 포함). γ=0은 over-smoothing, γ=2는 N2N + bias 보정.

In [ ]:
def train_nb2nb(noisy_t: torch.Tensor, clean_ref: NDArray, gamma: float = 2.0,
                 n_iters: int = 2000, lr: float = 1e-3, log_every: int = 200) -> tuple[nn.Module, dict]:
    """Train a TinyCNN with the Neighbor2Neighbor loss.

    Args:
        noisy_t: Single noisy image as (1, 1, H, W) tensor.
        clean_ref: Clean reference numpy array for monitoring.
        gamma: Regulariser weight.
        n_iters: Number of optimiser steps.
        lr: Adam learning rate.
        log_every: Logging interval.

    Returns:
        (trained_model, history).
    """
    model = TinyCNN(channels=32).to(noisy_t.device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'iter': [], 'rec_loss': [], 'reg_loss': [], 'gt_psnr': []}
    for it in range(n_iters):
        model.train()
        # Random sub-sampling for the noisy pair
        torch.manual_seed(it)
        g1_in, g2_in = neighbour_subsample(noisy_t)
        # Forward on g1(y) — has gradient
        f_g1 = model(g1_in)
        # Forward on full noisy y — NO gradient (Algorithm 1)
        with torch.no_grad():
            f_y = model(noisy_t)
            torch.manual_seed(it)
            g1_fy, g2_fy = neighbour_subsample(f_y)
        rec = F.mse_loss(f_g1, g2_in)
        reg = F.mse_loss(f_g1 - g2_in, g1_fy - g2_fy)
        loss = rec + gamma * reg
        opt.zero_grad(); loss.backward(); opt.step()

        if (it + 1) % log_every == 0:
            model.eval()
            with torch.no_grad():
                pred = np.clip(model(noisy_t).cpu().numpy()[0, 0], 0, 1)
            history['iter'].append(it + 1)
            history['rec_loss'].append(rec.item())
            history['reg_loss'].append(reg.item())
            history['gt_psnr'].append(psnr(pred, clean_ref))
            print(f'iter {it+1:>5d}  rec {rec.item():.5f}  reg {reg.item():.5f}  GT PSNR {history["gt_psnr"][-1]:.2f} dB')
    return model, history


print('=== Training γ = 0 (rec only) ===')
model_g0, hist_g0 = train_nb2nb(noisy_t, clean, gamma=0.0)
print('\n=== Training γ = 2 (with regulariser) ===')
model_g2, hist_g2 = train_nb2nb(noisy_t, clean, gamma=2.0)

---

## Part 6: Compare γ=0 vs γ=2 vs BM3D vs noisy / γ별 비교

In [ ]:
def predict(model: nn.Module, x: torch.Tensor) -> NDArray:
    """Single-pass inference, returns clipped numpy array."""
    model.eval()
    with torch.no_grad():
        return np.clip(model(x).cpu().numpy()[0, 0], 0, 1)


pred_g0 = predict(model_g0, noisy_t)
pred_g2 = predict(model_g2, noisy_t)

# BM3D baseline
try:
    import bm3d
    pred_bm3d = bm3d.bm3d(noisy, sigma_psd=sigma, stage_arg=bm3d.BM3DStages.ALL_STAGES)
    have_bm3d = True
except ImportError:
    pred_bm3d = None
    have_bm3d = False
    print('Note: bm3d package not installed; skipping BM3D comparison.')

results = {
    'noisy'           : psnr(noisy, clean),
    'Nb2Nb γ=0 (no reg)': psnr(pred_g0, clean),
    'Nb2Nb γ=2 (reg)'   : psnr(pred_g2, clean),
}
if have_bm3d:
    results['BM3D'] = psnr(pred_bm3d, clean)

print('PSNR comparison (dB):')
for k, v in results.items():
    print(f'  {k:<20s} : {v:.2f} dB')
print('\n→ γ=2 should beat γ=0 by ~0.3-0.7 dB (regulariser prevents over-smoothing).')

# Visual comparison
n_cols = 5 if have_bm3d else 4
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
axes[0].imshow(clean, cmap='gray', vmin=0, vmax=1); axes[0].set_title('clean')
axes[1].imshow(noisy, cmap='gray', vmin=0, vmax=1); axes[1].set_title(f'noisy ({results["noisy"]:.2f} dB)')
axes[2].imshow(pred_g0, cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Nb2Nb γ=0 ({results["Nb2Nb γ=0 (no reg)"]:.2f} dB)')
axes[3].imshow(pred_g2, cmap='gray', vmin=0, vmax=1)
axes[3].set_title(f'Nb2Nb γ=2 ({results["Nb2Nb γ=2 (reg)"]:.2f} dB)')
if have_bm3d:
    axes[4].imshow(pred_bm3d, cmap='gray', vmin=0, vmax=1)
    axes[4].set_title(f'BM3D ({results["BM3D"]:.2f} dB)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---

## Part 7: γ ablation curve / γ 어블레이션

Sweep \\( \gamma \in \{0, 0.5, 1, 2, 5, 10\} \\). Reproduces Table 3 / Fig. 6 of paper at small scale.

γ를 sweep해 성능 곡선 확인. γ=0 (over-smooth) → γ≈2 (sweet spot) → γ 큼 (under-smooth).

In [ ]:
gammas = [0.0, 0.5, 1.0, 2.0, 5.0, 10.0]
ablation = []
for g in gammas:
    m, _ = train_nb2nb(noisy_t, clean, gamma=g, n_iters=1500, log_every=10**6)  # silent
    p = predict(m, noisy_t)
    ablation.append(psnr(p, clean))
    print(f'γ = {g:5.1f}  PSNR = {ablation[-1]:.2f} dB')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gammas, ablation, '-o', color='C0')
ax.set_xlabel('γ (regulariser weight)'); ax.set_ylabel('GT PSNR (dB)')
ax.set_title('Nb2Nb regulariser ablation (cf. Table 3, Fig. 6)'); ax.grid(True)
if have_bm3d:
    ax.axhline(results['BM3D'], color='C1', linestyle='--', label='BM3D')
    ax.legend()
plt.tight_layout(); plt.show()

---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 |
|---|---|---|
| Random neighbour sub-sampler | §4.1 + Fig. 2 | `neighbour_subsample()` |
| Noise near-independence | Theorem 1 assumption | corr(n_g1, n_g2) ≈ 0 |
| Signal near-equality (small ε) | §3.2 | std(ε) small in flat regions |
| Reconstruction loss | Eq. 7 first term | `F.mse_loss(f_g1, g2)` |
| Regulariser term | Eq. 7 second term | `F.mse_loss(f_g1 - g2, g1_fy - g2_fy)` |
| Stop-gradient on outputs | §4.2 | `with torch.no_grad():` for `f_y` |
| γ sweet spot ≈ 2 | Table 3 / Fig. 6 | γ ablation curve in Part 7 |
| Single-pass inference | §5.1 | `predict()` (no ensemble) |

### Connections to other papers / 다른 논문과의 연결

- **#16 Noise2Noise** — Nb2Nb's loss is N2N applied to a sub-sampled pair from a single image; Theorem 1 generalises Lehtinen's result to non-zero signal gap.
- **#17 Noise2Void** / **#18 Noise2Self** — both use blind-spot / J-invariance via the *network*; Nb2Nb does it via the *data* (sub-sampling).
- **#19 Self2Self** — uses Bernoulli sampling + dropout ensemble; Nb2Nb uses spatial sub-sampling + regulariser, single-pass inference.
- **#7 BM3D** — non-learning baseline; Nb2Nb beats BM3D on synthetic Gaussian/Poisson and on real-world SIDD.
- **#22 Blind2Unblind** — direct successor (visible blind spots).

### Take-away / 결론

**한국어** — 작은 CNN으로도 Nb2Nb의 핵심 메커니즘이 작동함을 확인. \\( 2\times 2 \\) 셀에서 인접 픽셀 두 개를 무작위로 골라 만든 sub-image 쌍은 잡음이 거의 독립이고 신호가 거의 동일 — Noise2Noise pair로 사용 가능. 단순 N2N 손실(γ=0)은 작은 신호 gap \\( \varepsilon \\) 때문에 over-smooth하나, regulariser(γ=2)는 이 bias를 상쇄해 PSNR을 회복. γ sweep curve가 Table 3의 U자형 패턴을 재현. 추론은 single forward pass — DIP/S2S 대비 매우 빠름.

**English** — Even a tiny CNN replicates Nb2Nb's mechanism: sub-sampled pairs from a single noisy image have near-independent noise and near-equal signal — usable as a Noise2Noise pair. Plain N2N loss (γ=0) over-smooths because of the small but non-zero signal gap ε; the regulariser (γ=2) cancels this bias and recovers PSNR. γ ablation curve reproduces the U-shape of Table 3. Inference is a single forward pass — much faster than DIP / Self2Self.